# SDT full-parameter DPO: training and evaluation

This notebook runs the complete 100-record demonstration pipeline: repository setup, GPU verification, preference-pair preparation, tests, **full-parameter DPO**, locked baseline and DPO evaluation, paired comparison, and optional Google Drive backup.

The 100 records are sufficient to verify that the pipeline works, but not to establish a reliable alignment improvement. Run the cells in order in a fresh GPU runtime.

## 1. Confirm that Colab assigned a GPU
Select **Runtime → Change runtime type → GPU** before running this cell.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "No CUDA GPU is available. Change the Colab runtime to GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone or update the repository
For a private repository, create a Colab secret named `GITHUB_TOKEN`, paste a GitHub token with read access, and enable notebook access to that secret. A public repository needs no token.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Rana-Ezzeddine/SDT.git"
REPO_DIR = Path("/content/SDT")

github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass

git = ["git"]
if github_token:
    git += ["-c", f"http.extraHeader=AUTHORIZATION: bearer {github_token}"]

if not REPO_DIR.exists():
    try:
        subprocess.run(git + ["clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Clone failed. If the repository is private, add a GITHUB_TOKEN Colab secret."
        ) from exc
elif (REPO_DIR / ".git").exists():
    subprocess.run(git + ["-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Use a fresh runtime.")

os.chdir(REPO_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Working directory:", Path.cwd())
subprocess.run(["git", "status", "--short", "--branch"], check=True)

## 2b. Apply the validated TRL compatibility fix
This makes the notebook safe even if the GitHub clone still contains the earlier trainer version. It removes the redundant uncached evaluation call that caused the completed run to stop, and replaces the deprecated warmup ratio with two warmup steps. The cell does nothing when the repository is already fixed.

In [ ]:
from pathlib import Path

train_path = Path("src/sdt_dpo/train.py")
train_code = train_path.read_text()
old_evaluate = (
    '    validation_metrics = trainer.evaluate(\n'
    '        eval_dataset=validation_dpo,\n'
    '        metric_key_prefix="validation",\n'
    '    )'
)
new_evaluate = '    validation_metrics = trainer.evaluate(metric_key_prefix="validation")'
train_code = train_code.replace(old_evaluate, new_evaluate)
train_code = train_code.replace(
    'warmup_ratio=float(config.get("warmup_ratio", 0.1)),',
    'warmup_steps=int(config.get("warmup_steps", 2)),',
)
train_path.write_text(train_code)

config_path = Path("configs/full.yaml")
config_text = config_path.read_text().replace("warmup_ratio: 0.10", "warmup_steps: 2")
config_path.write_text(config_text)

evaluate_path = Path("src/sdt_dpo/evaluate.py")
evaluate_code = evaluate_path.read_text()
sequence_start = evaluate_code.index("def _sequence(")
sequence_end = evaluate_code.index("\n\ndef _score_response", sequence_start)
new_sequence = '''def _sequence(tokenizer: Any, prompt: str, response: str) -> tuple[list[int], int]:
    if not response:
        raise ValueError("empty_response")
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError("fast_tokenizer_required_for_response_offsets")

    marker = "<|sdt_response_boundary|>"
    while marker in prompt or marker in response:
        marker += "_"
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": marker + response},
    ]
    rendered_with_marker = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    marker_start = rendered_with_marker.find(marker)
    if marker_start < 0 or rendered_with_marker.count(marker) != 1:
        raise ValueError("response_boundary_marker_not_found")
    rendered = (
        rendered_with_marker[:marker_start]
        + rendered_with_marker[marker_start + len(marker) :]
    )
    encoded = tokenizer(
        rendered, add_special_tokens=False, return_offsets_mapping=True
    )
    full = list(encoded["input_ids"])
    offsets = list(encoded["offset_mapping"])
    response_start = next(
        (index for index, (start, end) in enumerate(offsets)
         if end > marker_start and end > start),
        None,
    )
    if response_start is None or response_start >= len(full):
        raise ValueError("empty_tokenized_response")
    return full, response_start'''
evaluate_path.write_text(
    evaluate_code[:sequence_start] + new_sequence + evaluate_code[sequence_end:]
)

evaluate_code = evaluate_path.read_text()
score_start = evaluate_code.index("def _score_response(")
score_end = evaluate_code.index("\n\ndef parse_args", score_start)
new_score = '''def _score_response(
    model: Any, torch: Any, token_ids: list[int], response_start: int, device: Any
) -> dict[str, float | int]:
    ids = torch.tensor([token_ids], dtype=torch.long, device=device)
    with torch.inference_mode():
        logits = model(input_ids=ids, use_cache=False).logits
        token_log_probs = torch.log_softmax(logits[:, :-1, :].float(), dim=-1)
        targets = ids[:, 1:]
        gathered = token_log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        response_log_probs = gathered[:, response_start - 1 :]
    return {
        "mean": float(response_log_probs.mean().item()),
        "sum": float(response_log_probs.sum().item()),
        "token_count": int(response_log_probs.numel()),
    }'''
evaluate_code = evaluate_code[:score_start] + new_score + evaluate_code[score_end:]
evaluate_code = evaluate_code.replace(
    '            margin = chosen_score - rejected_score',
    '            margin = float(chosen_score["mean"]) - float(rejected_score["mean"])',
)
evaluate_code = evaluate_code.replace(
    '                    "chosen_logprob_per_token": chosen_score,\n'
    '                    "rejected_logprob_per_token": rejected_score,',
    '                    "chosen_logprob_per_token": chosen_score["mean"],\n'
    '                    "rejected_logprob_per_token": rejected_score["mean"],\n'
    '                    "chosen_logprob_sum": chosen_score["sum"],\n'
    '                    "rejected_logprob_sum": rejected_score["sum"],\n'
    '                    "chosen_token_count": chosen_score["token_count"],\n'
    '                    "rejected_token_count": rejected_score["token_count"],',
)
evaluate_path.write_text(evaluate_code)

assert 'trainer.evaluate(metric_key_prefix="validation")' in train_path.read_text()
assert "warmup_steps: 2" in config_path.read_text()
assert "warmup_ratio" not in config_path.read_text()
assert "return_offsets_mapping=True" in evaluate_path.read_text()
assert "chat_template_prefix_mismatch" not in evaluate_path.read_text()
assert "chosen_logprob_sum" in evaluate_path.read_text()
print("Trainer and evaluation compatibility fixes confirmed.")

## 3. Install the project
The public Qwen checkpoint will be downloaded automatically later; no Hugging Face token is required.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 4. Inspect the full-DPO configuration
The learning rate is intentionally much lower than a LoRA learning rate. `precompute_ref_log_probs` reduces memory use by caching the frozen reference model's scores before training.

In [ ]:
from pathlib import Path

print(Path("configs/full.yaml").read_text())

## 5. Prepare the chosen/rejected pairs
Failed judge blocks are excluded rather than changed to score zero. The split is assigned at the prompt level before creating multiple response pairs.

In [ ]:
import subprocess

subprocess.run([
    "sdt-build-pairs",
    "--input", "data/raw/sdt_100_llama.json",
    "--output", "data/processed/dpo_pairs.jsonl",
    "--report", "data/processed/pair_report.json",
    "--min-common-judges", "2",
    "--min-margin", "0.10",
    "--min-confidence", "0.60",
    "--train-share", "0.70",
    "--validation-share", "0.15",
    "--seed", "42",
], check=True)

In [ ]:
import json
from pathlib import Path

pair_report = json.loads(Path("data/processed/pair_report.json").read_text())
summary_fields = [
    "records", "possible_pairs", "retained_pairs", "retained_prompts",
    "retained_by_split", "retained_by_confidence",
    "retained_by_comparison_type", "exclusion_reasons",
]
for field in summary_fields:
    print(f"{field}: {pair_report[field]}")

assert pair_report["retained_by_split"] == {"train": 139, "validation": 27, "test": 22}
print("Sample preparation counts match the audited pipeline.")

## 6. Run repository tests
These tests cover failed-judge handling, pair direction, prompt leakage, full-parameter configuration, and comparison statistics.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## 7. Train the full-parameter DPO model
This downloads `Qwen/Qwen2.5-0.5B-Instruct`, verifies that all 494M parameters are trainable, precomputes reference log-probabilities, trains for one epoch, validates, and saves a complete model. Expect several minutes on a Colab GPU.

Do not use the test results to change training settings after this point. If CUDA runs out of memory, start a fresh runtime, change `max_length: 1024` to `512`, and rerun from the beginning.

In [ ]:
import subprocess

subprocess.run(["sdt-train-dpo", "--config", "configs/full.yaml"], check=True)

## 8. Locate and verify the trained model
A successful fixed script saves the final model in `outputs/dpo-full`. The checkpoint fallback also lets this notebook recover a model from a run that completed training but stopped during final reporting.

In [ ]:
from pathlib import Path

final_model = Path("outputs/dpo-full")
has_final_weights = (final_model / "config.json").exists() and any(final_model.glob("*.safetensors"))

if has_final_weights:
    DPO_MODEL = str(final_model)
else:
    checkpoints = sorted(
        final_model.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    assert checkpoints, "Training produced neither a final model nor a checkpoint."
    DPO_MODEL = str(checkpoints[-1])

print("DPO model for evaluation:", DPO_MODEL)
print("Model files:")
for path in sorted(Path(DPO_MODEL).glob("*")):
    if path.is_file():
        print(" -", path.name)

## 9. Evaluate the unchanged baseline on the locked test pairs
The evaluator calculates length-normalized conditional response log-probability. A pair is correct when the model assigns a higher score to the judge-preferred response.

In [ ]:
import subprocess

subprocess.run([
    "sdt-evaluate-pairs",
    "--pairs", "data/processed/dpo_pairs.jsonl",
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--split", "test",
    "--max-length", "1024",
    "--output", "outputs/base-test.json",
    "--details", "outputs/base-test-pairs.jsonl",
], check=True)

## 10. Evaluate the full-DPO model on exactly the same test pairs

In [ ]:
import subprocess

subprocess.run([
    "sdt-evaluate-pairs",
    "--pairs", "data/processed/dpo_pairs.jsonl",
    "--model", DPO_MODEL,
    "--split", "test",
    "--max-length", "1024",
    "--output", "outputs/dpo-test.json",
    "--details", "outputs/dpo-test-pairs.jsonl",
], check=True)

## 11. Verify that evaluation is paired
This guard prevents an invalid comparison if either evaluator skipped every pair or the files came from different data splits.

In [ ]:
import json
from pathlib import Path

def load_details(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

base_rows = load_details("outputs/base-test-pairs.jsonl")
dpo_rows = load_details("outputs/dpo-test-pairs.jsonl")
base_ids = {str(row["pair_id"]) for row in base_rows}
dpo_ids = {str(row["pair_id"]) for row in dpo_rows}
shared_ids = base_ids & dpo_ids

print("Baseline evaluated pairs:", len(base_ids))
print("DPO evaluated pairs:", len(dpo_ids))
print("Shared pair IDs:", len(shared_ids))

if not shared_ids:
    for summary_path in ["outputs/base-test.json", "outputs/dpo-test.json"]:
        report = json.loads(Path(summary_path).read_text())
        print(summary_path, "overall=", report.get("overall"), "skipped=", report.get("skipped"))
    raise RuntimeError("No shared test pairs. Inspect the printed skipped reasons before comparing.")

assert base_ids == dpo_ids, "The evaluators did not score exactly the same pair IDs."
print("Pairing check passed.")

## 12. Produce the paired baseline-versus-DPO report

In [ ]:
import json
import random
import statistics
from collections import defaultdict
from pathlib import Path

BETA = 0.10
BOOTSTRAP_SAMPLES = 10_000
base = {str(row["pair_id"]): row for row in base_rows}
dpo = {str(row["pair_id"]): row for row in dpo_rows}
assert set(base) == set(dpo) and base, "Evaluations must have identical nonempty pair IDs."

relative_rows = []
for pair_id in sorted(base):
    before, after = base[pair_id], dpo[pair_id]
    required = {"chosen_logprob_sum", "rejected_logprob_sum",
                "chosen_token_count", "rejected_token_count"}
    assert required <= set(before) and required <= set(after), (
        "Rerun Sections 9 and 10 with the updated evaluator."
    )
    assert before["chosen_token_count"] == after["chosen_token_count"]
    assert before["rejected_token_count"] == after["rejected_token_count"]
    chosen_ratio = after["chosen_logprob_sum"] - before["chosen_logprob_sum"]
    rejected_ratio = after["rejected_logprob_sum"] - before["rejected_logprob_sum"]
    implicit_margin = BETA * (chosen_ratio - rejected_ratio)
    relative_rows.append({
        "pair_id": pair_id, "prompt_id": str(before["prompt_id"]),
        "baseline_correct": bool(before["correct"]),
        "dpo_correct": bool(after["correct"]),
        "absolute_accuracy_delta": float(after["correct"]) - float(before["correct"]),
        "baseline_model_margin": float(before["model_margin"]),
        "dpo_model_margin": float(after["model_margin"]),
        "chosen_policy_logratio": chosen_ratio,
        "rejected_policy_logratio": rejected_ratio,
        "dpo_implicit_reward_margin": implicit_margin,
        "dpo_implicit_reward_correct": implicit_margin > 0,
        "checkpoint_behavior_changed": abs(chosen_ratio) > 1e-8 or abs(rejected_ratio) > 1e-8,
    })

def prompt_macro(rows, field):
    groups = defaultdict(list)
    for row in rows: groups[row["prompt_id"]].append(float(row[field]))
    return statistics.fmean(statistics.fmean(values) for values in groups.values())

def prompt_cluster_interval(rows, field, seed=42):
    groups = defaultdict(list)
    for row in rows: groups[row["prompt_id"]].append(float(row[field]))
    values = [statistics.fmean(group) for group in groups.values()]
    rng = random.Random(seed)
    estimates = sorted(statistics.fmean(rng.choice(values) for _ in values)
                       for _ in range(BOOTSTRAP_SAMPLES))
    return [estimates[int(.025*(len(estimates)-1))], estimates[int(.975*(len(estimates)-1))]]

implicit_accuracy = statistics.fmean(float(r["dpo_implicit_reward_correct"]) for r in relative_rows)
base_accuracy = statistics.fmean(float(r["baseline_correct"]) for r in relative_rows)
dpo_accuracy = statistics.fmean(float(r["dpo_correct"]) for r in relative_rows)
implicit_margins = [r["dpo_implicit_reward_margin"] for r in relative_rows]
comparison = {
    "paired_n": len(relative_rows),
    "unique_prompts": len({r["prompt_id"] for r in relative_rows}),
    "beta": BETA,
    "primary_dpo_relative_metrics": {
        "implicit_reward_accuracy": implicit_accuracy,
        "implicit_reward_prompt_macro_accuracy": prompt_macro(relative_rows, "dpo_implicit_reward_correct"),
        "implicit_reward_prompt_cluster_accuracy_95": prompt_cluster_interval(relative_rows, "dpo_implicit_reward_correct"),
        "mean_implicit_reward_margin": statistics.fmean(implicit_margins),
        "median_implicit_reward_margin": statistics.median(implicit_margins),
        "implicit_reward_margin_prompt_cluster_95": prompt_cluster_interval(relative_rows, "dpo_implicit_reward_margin", 43),
    },
    "secondary_absolute_likelihood_metrics": {
        "baseline_preference_accuracy": base_accuracy,
        "dpo_preference_accuracy": dpo_accuracy,
        "accuracy_delta_dpo_minus_baseline": dpo_accuracy - base_accuracy,
        "baseline_prompt_macro_accuracy": prompt_macro(relative_rows, "baseline_correct"),
        "dpo_prompt_macro_accuracy": prompt_macro(relative_rows, "dpo_correct"),
        "prompt_macro_accuracy_delta": prompt_macro(relative_rows, "absolute_accuracy_delta"),
        "mean_model_margin_baseline": statistics.fmean(r["baseline_model_margin"] for r in relative_rows),
        "mean_model_margin_dpo": statistics.fmean(r["dpo_model_margin"] for r in relative_rows),
    },
    "checkpoint_behavior_check": {
        "pairs_with_changed_completion_logprobs": sum(r["checkpoint_behavior_changed"] for r in relative_rows),
        "all_pairs_changed": all(r["checkpoint_behavior_changed"] for r in relative_rows),
        "mean_abs_chosen_policy_logratio": statistics.fmean(abs(r["chosen_policy_logratio"]) for r in relative_rows),
        "mean_abs_rejected_policy_logratio": statistics.fmean(abs(r["rejected_policy_logratio"]) for r in relative_rows),
    },
}
Path("outputs/base-vs-dpo.json").write_text(json.dumps(comparison, indent=2) + "\n")
with Path("outputs/dpo-relative-test-pairs.jsonl").open("w") as handle:
    for row in relative_rows: handle.write(json.dumps(row) + "\n")
print(json.dumps(comparison, indent=2))

## 13. Display the main results
The DPO-relative implicit reward accuracy and margin are primary because they measure the objective DPO optimized. Absolute likelihood ranking remains a secondary diagnostic. Prompt-cluster intervals account for multiple pairs from one prompt; with only 10 test prompts, expect wide uncertainty.

In [ ]:
import json
import pandas as pd
from pathlib import Path

comparison = json.loads(Path("outputs/base-vs-dpo.json").read_text())
primary = comparison["primary_dpo_relative_metrics"]
secondary = comparison["secondary_absolute_likelihood_metrics"]
behavior = comparison["checkpoint_behavior_check"]
table = pd.DataFrame([
    {
        "model": "Baseline",
        "absolute_pair_accuracy": secondary["baseline_preference_accuracy"],
        "absolute_prompt_macro_accuracy": secondary["baseline_prompt_macro_accuracy"],
        "mean_absolute_margin": secondary["mean_model_margin_baseline"],
    },
    {
        "model": "Full DPO",
        "absolute_pair_accuracy": secondary["dpo_preference_accuracy"],
        "absolute_prompt_macro_accuracy": secondary["dpo_prompt_macro_accuracy"],
        "mean_absolute_margin": secondary["mean_model_margin_dpo"],
    },
])
print(table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("\nPRIMARY DPO-RELATIVE RESULTS")
print("Implicit reward accuracy:", primary["implicit_reward_accuracy"])
print("Prompt-macro implicit accuracy:", primary["implicit_reward_prompt_macro_accuracy"])
print("Prompt-cluster accuracy 95% interval:", primary["implicit_reward_prompt_cluster_accuracy_95"])
print("Mean implicit reward margin:", primary["mean_implicit_reward_margin"])
print("Margin prompt-cluster 95% interval:", primary["implicit_reward_margin_prompt_cluster_95"])
print("\nSECONDARY ABSOLUTE RESULTS")
print("Absolute pair-accuracy delta:", secondary["accuracy_delta_dpo_minus_baseline"])
print("Absolute prompt-macro delta:", secondary["prompt_macro_accuracy_delta"])
print("\nCHECKPOINT BEHAVIOR CHECK")
print(behavior)

## 14. Save the complete outputs to Google Drive
Colab runtime storage is temporary. Run this cell after evaluation to preserve the full checkpoint and reports. Each run gets a timestamped directory, so previous results are not overwritten.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
from google.colab import drive

drive.mount("/content/drive")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destination = Path("/content/drive/MyDrive/SDT_full_DPO_results") / timestamp
shutil.copytree("outputs", destination)
print("Saved outputs to:", destination)

## Optional appendix: deliberate tiny-overfit canary
This diagnostic is separate from the main experiment and does not use the test set. It trains on only eight training pairs and deliberately evaluates on those same pairs. A working pipeline should be able to drive its training-pair DPO reward accuracy upward. Leave the flag `False` during a normal run.

In [ ]:
RUN_OVERFIT_SANITY = False

if RUN_OVERFIT_SANITY:
    import subprocess
    config = Path("configs/sanity_overfit.yaml")
    assert config.exists(), "Pull the updated repository containing sanity_overfit.yaml."
    subprocess.run(["sdt-train-dpo", "--config", str(config)], check=True)
    metrics = json.loads(Path("outputs/dpo-overfit-sanity/validation_metrics.json").read_text())
    print(json.dumps(metrics, indent=2))
else:
    print("Tiny-overfit canary skipped (normal).")